In [ ]:
!pip install opencv-python
import os
import numpy as np
import cv2
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

# CLAHE PAPER

In [ ]:
"""
Branch B (CLAHE) Preprocessing Pipeline for Diabetic Retinopathy Grading
==========================================================================
MODIFIED VERSION — transparent background instead of black padding.

Based on Section 3.3 of:
"Dual-Resolution Attention-Gated Deep Learning with Ordinal Regression
for Diabetic Retinopathy Grading" (Hashmi, 2026)

Perubahan dari versi asli:
  - `circular_crop` sekarang mengembalikan gambar RGBA. Area di luar
    lingkaran retina diberi alpha=0 (transparan), BUKAN dihitamkan.
  - Karena CLAHE bekerja di ruang warna LAB (3 channel), CLAHE tetap
    dijalankan hanya pada channel RGB; channel alpha disisipkan kembali
    setelah CLAHE selesai, tidak ikut diproses.
  - Semua fungsi penyimpanan (`preprocess_folder_branch_b`) sekarang
    SELALU menyimpan sebagai .png (bukan mengikuti ekstensi asli),
    karena JPG tidak mendukung transparansi dan akan memaksa alpha
    dibuang / diisi warna solid.

Catatan penting soal geometri:
  Sebuah lingkaran tidak mungkin mengisi penuh sebuah gambar berbentuk
  persegi/persegi panjang — sudut-sudutnya pasti berada di luar lingkaran.
  Yang bisa diubah bukan "menghilangkan sudut" (tidak mungkin selama file
  tetap berbentuk rectangular), tapi MENGGANTI warna sudut itu dari hitam
  solid menjadi transparan, sehingga saat dibuka di software yang mendukung
  alpha (Photoshop, GIMP, browser, dsb.) sudut itu tidak terlihat sebagai
  warna latar apa pun.

Run this in Google Colab. Install deps first with:
    !pip install opencv-python-headless scikit-image tqdm
"""

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
CLAHE_CLIP_LIMIT = 2.0       # standard CLAHE clip limit
CLAHE_TILE_GRID = (8, 8)     # local tile size for CLAHE


# --------------------------------------------------------------------------
# Stage 1 — Retinal region isolation (Section 3.3.1) — now RGBA output
# --------------------------------------------------------------------------
def circular_crop(image, threshold=10, pad=0):
    """
    Crop the image circularly to the retinal disc, removing the
    uninformative dark border around the fundus photo.

    Returns an RGBA image: pixels outside the inscribed circle have
    alpha=0 (transparent) instead of being painted black. Pixels inside
    the circle have alpha=255.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    mask = gray > threshold

    if mask.sum() == 0:
        # Degenerate/blank image — return as-is (fully opaque) rather than crash.
        h, w = image.shape[:2]
        alpha = np.full((h, w), 255, dtype=np.uint8)
        return np.dstack([image, alpha])

    coords = np.argwhere(mask)
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1

    y0, x0 = max(y0 - pad, 0), max(x0 - pad, 0)
    y1, x1 = min(y1 + pad, image.shape[0]), min(x1 + pad, image.shape[1])

    cropped = image[y0:y1, x0:x1]

    # Make it square (retina is circular) so downstream resize doesn't distort it.
    # The padding needed to square it off is transparent, not black.
    h, w = cropped.shape[:2]
    side = max(h, w)
    square_rgb = np.zeros((side, side, 3), dtype=cropped.dtype)
    y_off, x_off = (side - h) // 2, (side - w) // 2
    square_rgb[y_off:y_off + h, x_off:x_off + w] = cropped

    # Circle mask: 255 = keep (opaque), 0 = outside retina (transparent)
    circ_mask = np.zeros((side, side), dtype=np.uint8)
    cv2.circle(circ_mask, (side // 2, side // 2), side // 2, 255, -1)

    square_rgb = cv2.bitwise_and(square_rgb, square_rgb, mask=circ_mask)

    # Alpha channel = circle mask directly -> transparent outside, opaque inside
    rgba = np.dstack([square_rgb, circ_mask])
    return rgba


# --------------------------------------------------------------------------
# Stage 2 — Histogram matching for domain alignment (Section 3.3.2)
# --------------------------------------------------------------------------
def histogram_match_to_reference(image_rgb, reference_image_rgb):
    """
    Align first-order intensity statistics of `image_rgb` (3-channel) to a
    fixed `reference_image_rgb`, channel-wise. Per the paper this is applied
    to Messidor-2 images only, against a single reference image drawn from
    the APTOS training split — never to APTOS images themselves, and never
    using test-set statistics.

    Operates on RGB (3 channel) only — call this BEFORE splitting off alpha,
    or pass image_rgb[..., :3].
    """
    matched = match_histograms(image_rgb, reference_image_rgb, channel_axis=-1)
    return np.clip(matched, 0, 255).astype(np.uint8)


# --------------------------------------------------------------------------
# Stage 3 — CLAHE enhancement (Section 3.3.2, Branch B)
# --------------------------------------------------------------------------
def apply_clahe(image_rgb, clip_limit=CLAHE_CLIP_LIMIT, tile_grid=CLAHE_TILE_GRID):
    """
    Contrast-Limited Adaptive Histogram Equalisation.
    Applied on the L channel of LAB colour space so contrast is boosted
    without distorting colour balance, then merged back to RGB.

    Operates on RGB (3 channel) only.
    """
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_enhanced = clahe.apply(l_channel)

    lab_enhanced = cv2.merge((l_enhanced, a_channel, b_channel))
    rgb_enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
    return rgb_enhanced


# --------------------------------------------------------------------------
# Full Branch B pipeline (no resize, no normalisation) — RGBA in/out
# --------------------------------------------------------------------------
def preprocess_branch_b(image_path, reference_image=None, return_intermediate=False):
    """
    Run the Branch B (CLAHE) preprocessing pipeline on a single image:
    circular crop -> (optional) histogram matching -> CLAHE.

    Returns
    -------
    clahe_rgba : np.ndarray, uint8, RGBA (H, W, 4)
        CLAHE-enhanced image at its native cropped resolution (not resized).
        Alpha channel is 0 outside the retinal circle, 255 inside.
    intermediates : dict (only if return_intermediate=True)
    """
    bgr = cv2.imread(image_path)
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    cropped_rgba = circular_crop(rgb)          # (H, W, 4)
    cropped_rgb = cropped_rgba[..., :3]
    alpha = cropped_rgba[..., 3]

    if reference_image is not None:
        matched_rgb = histogram_match_to_reference(cropped_rgb, reference_image)
    else:
        matched_rgb = cropped_rgb

    clahe_rgb = apply_clahe(matched_rgb)
    clahe_rgba = np.dstack([clahe_rgb, alpha])   # re-attach untouched alpha

    if return_intermediate:
        return clahe_rgba, {
            "original": rgb,
            "circle_cropped": cropped_rgba,
            "histogram_matched": matched_rgb,
            "clahe": clahe_rgba,
        }
    return clahe_rgba


# --------------------------------------------------------------------------
# Batch helper — always saves as PNG (required for transparency)
# --------------------------------------------------------------------------
def preprocess_folder_branch_b(input_folder, output_folder=None, reference_image=None,
                                extensions=(".png", ".jpg", ".jpeg"), verbose=True):
    """
    Preprocess every image in `input_folder` through the Branch B pipeline
    (circle crop -> optional histogram match -> CLAHE).

    Output files are ALWAYS saved as .png regardless of the source
    extension, because PNG is the only common format here that preserves
    the transparent alpha channel.

    Returns
    -------
    results : dict {filename_without_ext + ".png": clahe_rgba_image}
    failed : list of str
    """
    if output_folder is not None:
        os.makedirs(output_folder, exist_ok=True)

    filenames = sorted(
        f for f in os.listdir(input_folder) if f.lower().endswith(extensions)
    )

    results = {}
    failed = []

    iterator = tqdm(filenames, desc="Branch B preprocessing", unit="img") if verbose else filenames

    for fname in iterator:
        in_path = os.path.join(input_folder, fname)
        try:
            clahe_rgba = preprocess_branch_b(in_path, reference_image=reference_image)
        except Exception as e:
            failed.append(fname)
            if verbose:
                iterator.write(f"FAILED: {fname} ({e})")
            continue

        out_name = os.path.splitext(fname)[0] + ".png"
        results[out_name] = clahe_rgba

        if output_folder is not None:
            out_path = os.path.join(output_folder, out_name)
            # cv2 expects BGR(A) order when writing
            bgra_out = cv2.cvtColor(clahe_rgba, cv2.COLOR_RGBA2BGRA)
            cv2.imwrite(out_path, bgra_out)  # .png extension -> PNG w/ alpha preserved

    if verbose:
        print(f"Done. {len(results)} succeeded, {len(failed)} failed.")
        if output_folder is not None:
            print(f"Saved to: {output_folder} (all as .png with transparent corners)")

    return results, failed


# --------------------------------------------------------------------------
# Visualisation helper (mirrors Figure 2 / Figure 3 in the paper)
# --------------------------------------------------------------------------
def show_pipeline(image_path, reference_image=None):
    """Display each stage of the Branch B pipeline side by side.
    RGBA stages are shown over a checkerboard-free white background just
    for viewing convenience; the saved PNG files keep real transparency.
    """
    _, stages = preprocess_branch_b(
        image_path, reference_image=reference_image, return_intermediate=True
    )

    def _for_display(img):
        if img.shape[-1] == 4:
            rgb, a = img[..., :3].astype(float), img[..., 3:4].astype(float) / 255.0
            white_bg = np.ones_like(rgb) * 255
            return (rgb * a + white_bg * (1 - a)).astype(np.uint8)
        return img

    titles = ["Original", "Circle crop (transparent bg)"]
    imgs = [stages["original"], stages["circle_cropped"]]
    if reference_image is not None:
        titles.append("Histogram matched")
        imgs.append(stages["histogram_matched"])
    titles.append("CLAHE (Branch B, transparent bg)")
    imgs.append(stages["clahe"])

    fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(_for_display(img))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Example usage (Colab)
# --------------------------------------------------------------------------
if __name__ == "__main__":
    # 1) APTOS image — no histogram matching:
    #    clahe_img = preprocess_branch_b("/content/aptos/0a4e1a29ffff.png")

    # 2) Messidor-2 image — histogram matched to a fixed APTOS reference,
    #    per Section 3.3.2 ("the first image of the shuffled APTOS
    #    training partition"):
    #    aptos_reference = cv2.cvtColor(
    #        cv2.imread("/content/aptos/REFERENCE_IMAGE.png"), cv2.COLOR_BGR2RGB
    #    )
    #    clahe_img = preprocess_branch_b(
    #        "/content/messidor2/img001.png", reference_image=aptos_reference
    #    )

    # 3) Visualise the pipeline stages for one image:
    #    show_pipeline("/content/aptos/0a4e1a29ffff.png")

    # 4) Batch-process a whole folder and save outputs to disk:
    #    APTOS (no histogram matching):
    #    results, failed = preprocess_folder_branch_b(
    #        input_folder="/content/aptos_val/",
    #        output_folder="/content/aptos_val_branchB/",
    #    )
    #
    #    Messidor-2 (histogram matched to a fixed APTOS reference):
    #    aptos_reference = cv2.cvtColor(
    #        cv2.imread("/content/aptos/REFERENCE_IMAGE.png"), cv2.COLOR_BGR2RGB
    #    )
    #    results, failed = preprocess_folder_branch_b(
    #        input_folder="/content/messidor2_test/",
    #        output_folder="/content/messidor2_test_branchB/",
    #        reference_image=aptos_reference,
    #    )

    print("Import this module and call preprocess_branch_b() / "
          "preprocess_folder_branch_b() / show_pipeline() on your own "
          "image paths — see the commented examples above.")

# CLAHE

In [ ]:
# ZOOM: dipakai HANYA kalau AUTO_TRIM = False (lihat di bawah).
# 1.0  = tidak zoom, lingkaran mepet ke sisi terpendek foto
# 1.3+ = zoom lebih dekat ke tengah
ZOOM = 1.3

# AUTO_TRIM: kalau True, script otomatis MENCARI posisi objek/subjek di
# dalam foto (dengan mendeteksi area yang BUKAN background gelap/hitam),
# lalu lingkaran dipasang pas mengikuti objek itu -- bukan cuma zoom ke
# tengah frame. Ini yang menyelesaikan masalah "background hitam masih
# ikut kepotong" kalau objeknya kecil / tidak di tengah persis.
AUTO_TRIM = True

# Seberapa gelap dianggap "background" (0-255). Pixel dengan brightness
# di bawah angka ini dianggap background dan diabaikan saat mencari
# posisi objek. Naikkan kalau background bukan hitam pekat (misal abu2 gelap).
BLACK_THRESHOLD = 20

# Padding tambahan di sekitar objek yang terdeteksi (1.0 = pas-pasan,
# 1.15 = ada sedikit ruang napas di sekitar objek). Naikkan kalau hasil
# crop terlalu ketat/mepet ke objek.
TRIM_PADDING = 1.0

# DEBUG: kalau True, script juga menyimpan versi foto dengan kotak merah
# di sekitar area objek yang terdeteksi, ke folder DEBUG_FOLDER di bawah.
# Berguna untuk mengecek apakah deteksi objeknya benar atau meleset.
DEBUG = False
#DEBUG_FOLDER = OUTPUT_FOLDER + "_debug"

# MIN_CONTENT_AREA_RATIO: batas minimum ukuran objek supaya dianggap
# "objek asli" (bukan noise). Nilainya rasio terhadap luas foto (0-1).
# Contoh: 0.01 = objek harus minimal 1% dari luas foto untuk terdeteksi.
# Kalau objek di foto kamu ada yang KECIL (misal cuma beberapa persen
# dari luas foto) dan ternyata tidak ke-detect, TURUNKAN angka ini,
# misal ke 0.002 (0.2%) atau lebih kecil lagi.
MIN_CONTENT_AREA_RATIO = 0.002

# ---------------------------------------------------------
# CLAHE (Contrast Limited Adaptive Histogram Equalization)
# ---------------------------------------------------------
# Dipakai SETELAH bounding box objek ditemukan & di-crop -- jadi
# urutannya: deteksi posisi objek dulu (seperti biasa) -> crop area
# objek itu -> BARU CLAHE diterapkan ke hasil crop tsb -> baru
# di-resize & dibentuk lingkaran. Gunanya menaikkan kontras lokal pada
# foto objek yang akan jadi hasil akhir (misal foto yang agak flat/pucat
# jadi lebih jelas detailnya), bukan untuk membantu proses deteksi.
#
# CLAHE diterapkan lewat channel L (lightness) di color space LAB,
# supaya kontras naik tapi warna (hue/saturation) tidak melenceng, dan
# channel alpha (transparansi) tidak ikut diutak-atik.
USE_CLAHE = True

# Seberapa kuat CLAHE menaikkan kontras. Makin tinggi = makin kontras,
# tapi juga makin gampang noise ikut terangkat. 2.0-4.0 biasanya aman.
CLAHE_CLIP_LIMIT = 2.5

# Ukuran grid tile CLAHE (lebar, tinggi) dalam pixel-grid, bukan pixel.
# Grid lebih kecil -> kontras lebih "lokal"/detail. Grid lebih besar ->
# efeknya lebih halus/global. 8x8 adalah default yang umum dipakai.
CLAHE_TILE_GRID_SIZE = (8, 8)

VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tiff", ".tif")


def apply_clahe_to_crop(image: Image.Image, clip_limit: float = 2.5,
                         tile_grid_size=(8, 8)) -> Image.Image:
    """
    Menerapkan CLAHE ke gambar hasil CROP (RGBA), yaitu potongan foto
    yang sudah dipas-kan ke area objek hasil deteksi bounding box.

    CLAHE diterapkan lewat channel L (lightness) di color space LAB:
    - Warna (hue/saturation, channel A & B di LAB) TIDAK ikut diubah,
      cuma kontras terangnya yang ditingkatkan secara lokal.
    - Channel alpha (transparansi) dipisah dulu, tidak ikut diproses,
      lalu digabung lagi setelah CLAHE selesai.

    Return gambar RGBA baru dengan kontras yang sudah ditingkatkan.
    """
    rgba = np.array(image.convert("RGBA"))
    rgb = rgba[:, :, :3]
    alpha = rgba[:, :, 3]

    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_channel = clahe.apply(l_channel)

    lab = cv2.merge((l_channel, a_channel, b_channel))
    rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    result = np.dstack((rgb, alpha))
    return Image.fromarray(result, "RGBA")


def get_content_bbox(image: Image.Image, threshold: int = 20):
    """
    Mencari bounding box objek utama di foto memakai deteksi contour
    (bukan cuma threshold biasa). Ini jauh lebih tahan terhadap noise
    dibanding metode getbbox() sederhana -- karena hanya BLOB TERBESAR
    yang dianggap objek, noise/artefak kecil di background diabaikan.

    Return (left, top, right, bottom) atau None kalau tidak ada objek
    yang cukup jelas terdeteksi.
    """
    arr = np.array(image.convert("L"))

    # Kurangi noise dulu sebelum threshold
    arr = cv2.GaussianBlur(arr, (5, 5), 0)

    _, bw = cv2.threshold(arr, threshold, 255, cv2.THRESH_BINARY)

    # Morphological opening: menghapus titik-titik noise kecil yang
    # ikut kedeteksi terang, tanpa menghapus objek utama yang besar
    kernel = np.ones((7, 7), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # Ambil HANYA contour (blob) TERBESAR -> ini objek utamanya
    largest = max(contours, key=cv2.contourArea)

    # Kalau blob terbesar terlalu kecil (cuma noise), anggap tidak ada objek
    img_area = arr.shape[0] * arr.shape[1]
    if cv2.contourArea(largest) < img_area * MIN_CONTENT_AREA_RATIO:
        return None

    x, y, w, h = cv2.boundingRect(largest)
    return (x, y, x + w, y + h)


def circular_crop(image: Image.Image, zoom: float = 1.0,
                   auto_trim: bool = True, threshold: int = 20,
                   padding: float = 1.15, use_clahe: bool = True,
                   clahe_clip_limit: float = 2.5,
                   clahe_tile_grid_size=(8, 8)) -> Image.Image:
    """
    Membuat crop lingkaran pada gambar, ukuran canvas OUTPUT tetap sama
    dengan aslinya (width x height tidak berubah).

    Kalau auto_trim=True: script mendeteksi dulu posisi objek/subjek di
    foto (area yang bukan background gelap), lalu lingkaran dipasang
    mengikuti posisi & ukuran objek itu (dengan sedikit padding).

    Kalau auto_trim=False atau objek tidak terdeteksi: fallback ke mode
    zoom manual dari tengah frame seperti sebelumnya.

    Kalau use_clahe=True: SETELAH area crop ditentukan (baik dari hasil
    auto_trim maupun fallback zoom), potongan foto tsb akan dilewatkan
    CLAHE dulu untuk menaikkan kontras lokalnya, baru kemudian di-resize
    dan dibentuk lingkaran.
    """
    image = image.convert("RGBA")
    width, height = image.size
    diameter = min(width, height)

    content_box = None
    if auto_trim:
        content_box = get_content_bbox(image, threshold=threshold)

    if content_box is not None:
        cl, ct, cr, cb = content_box
        content_w = cr - cl
        content_h = cb - ct
        content_cx = cl + content_w / 2
        content_cy = ct + content_h / 2

        # Ukuran crop = sisi terpanjang objek, dikasih padding
        crop_size = int(max(content_w, content_h) * padding)
        crop_size = max(1, min(crop_size, width, height))  # jangan lebih besar dari foto

        left = int(content_cx - crop_size / 2)
        top = int(content_cy - crop_size / 2)

        # Pastikan area crop tidak keluar dari batas foto
        left = max(0, min(left, width - crop_size))
        top = max(0, min(top, height - crop_size))
        right = left + crop_size
        bottom = top + crop_size
    else:
        # Fallback: mode zoom manual dari tengah frame
        crop_size = max(1, int(diameter / zoom))
        left = (width - crop_size) // 2
        top = (height - crop_size) // 2
        right = left + crop_size
        bottom = top + crop_size

    # Ambil area yang sudah ditentukan (ini foto objek yang akan jadi hasil akhir)
    center_crop = image.crop((left, top, right, bottom))

    # CLAHE diterapkan DI SINI -- setelah crop objek didapat, sebelum resize.
    if use_clahe:
        center_crop = apply_clahe_to_crop(
            center_crop,
            clip_limit=clahe_clip_limit,
            tile_grid_size=clahe_tile_grid_size,
        )

    # Resize supaya pas mengisi diameter lingkaran
    center_crop = center_crop.resize((diameter, diameter), Image.LANCZOS)

    # Buat mask lingkaran seukuran diameter
    mask = Image.new("L", (diameter, diameter), 0)
    draw = ImageDraw.Draw(mask)
    draw.ellipse((0, 0, diameter, diameter), fill=255)

    circle_img = Image.new("RGBA", (diameter, diameter), (0, 0, 0, 0))
    circle_img.paste(center_crop, (0, 0), mask=mask)

    # Tempel lingkaran ke canvas seukuran foto ASLI (ukuran tidak berubah).
    # PENTING: posisi tempelnya harus PAS di tengah OBJEK yang terdeteksi
    # (left/top di atas), BUKAN di tengah kanvas -- supaya kalau objeknya
    # tadinya tidak center di foto, hasil lingkarannya tetap sesuai posisi
    # objek yang sebenarnya, bukan meloncat ke tengah frame.
    result = Image.new("RGBA", (width, height), (0, 0, 0, 0))

    # Titik tengah area crop asli (left/top/right/bottom di atas)
    crop_center_x = left + crop_size / 2
    crop_center_y = top + crop_size / 2

    paste_x = int(round(crop_center_x - diameter / 2))
    paste_y = int(round(crop_center_y - diameter / 2))

    # Pastikan lingkaran tidak keluar dari batas kanvas
    paste_x = max(0, min(paste_x, width - diameter))
    paste_y = max(0, min(paste_y, height - diameter))

    result.paste(circle_img, (paste_x, paste_y), mask=circle_img)

    return result


def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    if DEBUG:
        os.makedirs(DEBUG_FOLDER, exist_ok=True)

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    failed_files = []   # list (filename, error_message) untuk yang gagal
    skipped_count = 0
    ok_count = 0

    # Progress bar tunggal untuk seluruh proses -- tidak print per foto lagi
    pbar = tqdm(files, desc="Processing", unit="img")

    for filename in pbar:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename_check = name_only + ".png"
        output_path_check = os.path.join(output_folder, output_filename_check)
        if os.path.exists(output_path_check):
            skipped_count += 1
            pbar.set_postfix(ok=ok_count, skip=skipped_count, fail=len(failed_files))
            continue

        try:
            with Image.open(input_path) as img:
                original_size = img.size

                if DEBUG:
                    box = get_content_bbox(img.convert("RGBA"), threshold=BLACK_THRESHOLD)
                    debug_img = img.convert("RGB").copy()
                    if box is not None:
                        d = ImageDraw.Draw(debug_img)
                        d.rectangle(box, outline=(255, 0, 0), width=5)
                    debug_img.save(os.path.join(DEBUG_FOLDER, name_only + "_debug.jpg"))

                cropped = circular_crop(
                    img,
                    zoom=ZOOM,
                    auto_trim=AUTO_TRIM,
                    threshold=BLACK_THRESHOLD,
                    padding=TRIM_PADDING,
                    use_clahe=USE_CLAHE,
                    clahe_clip_limit=CLAHE_CLIP_LIMIT,
                    clahe_tile_grid_size=CLAHE_TILE_GRID_SIZE,
                )

                assert cropped.size == original_size, "Ukuran berubah!"

                output_filename = name_only + ".png"
                output_path = os.path.join(output_folder, output_filename)

                cropped.save(output_path, "PNG")
                ok_count += 1

        except OSError as e:
            if e.errno == 107:
                # Mount Drive terputus -> file lain di belakangnya juga
                # akan gagal semua. Hentikan proses biar tidak spam GAGAL.
                failed_files.append((filename, f"Koneksi Google Drive terputus ({e})"))
                pbar.set_postfix(ok=ok_count, skip=skipped_count, fail=len(failed_files))
                pbar.close()
                print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
                print(">> lalu jalankan CELL ini lagi (file yang sudah OK tidak perlu diulang).")
                break
            failed_files.append((filename, str(e)))
        except Exception as e:
            failed_files.append((filename, str(e)))

        pbar.set_postfix(ok=ok_count, skip=skipped_count, fail=len(failed_files))

    pbar.close()

    # ------------------- RINGKASAN AKHIR -------------------
    print(f"\nSelesai. {ok_count} berhasil, {skipped_count} dilewati (sudah ada), "
          f"{len(failed_files)} gagal.")
    print(f"Hasil tersimpan di: {output_folder}")

    if failed_files:
        print(f"\nDaftar file yang GAGAL diproses ({len(failed_files)}):")
        for fname, err in failed_files:
            print(f"  - {fname}: {err}")
    else:
        print("\nTidak ada file yang gagal. 🎉")

# CLAHE DOANG

In [ ]:
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install opencv-python
import cv2
import os
from tqdm import tqdm

VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


def apply_clahe_fundus(image_path):
    """Apply CLAHE (in LAB space) to a single fundus image and return the enhanced BGR image."""
    img = cv2.imread(image_path)
    if img is None:
        return None

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    enhanced_bgr = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    return enhanced_bgr


def process_folder(input_folder: str, output_folder: str):
    if not os.path.isdir(input_folder):
        print(f"Folder input tidak ditemukan: {input_folder}")
        print("Pastikan path-nya sudah benar (cek lewat panel Files di sidebar Colab).")
        return

    os.makedirs(output_folder, exist_ok=True)

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    if not files:
        print(f"Tidak ada gambar ditemukan di: {input_folder}")
        return

    print(f"Ditemukan {len(files)} gambar. Memproses...\n")

    failed_files = []
    drive_disconnected = False

    progress = tqdm(files, desc="Memproses gambar", unit="gambar")

    for filename in progress:
        input_path = os.path.join(input_folder, filename)
        name_only, _ext = os.path.splitext(filename)

        output_filename = name_only + ".png"
        output_path = os.path.join(output_folder, output_filename)
        if os.path.exists(output_path):
            continue

        try:
            enhanced = apply_clahe_fundus(input_path)
            if enhanced is None:
                failed_files.append((filename, "Gagal dibaca (bukan gambar valid)"))
                continue

            cv2.imwrite(output_path, enhanced)

        except OSError as e:
            if e.errno == 107:
                failed_files.append((filename, "Koneksi Google Drive terputus"))
                drive_disconnected = True
                break
            failed_files.append((filename, str(e)))
        except Exception as e:
            failed_files.append((filename, str(e)))

    progress.close()

    print(f"\nSelesai. Hasil tersimpan di: {output_folder}")

    if failed_files:
        print(f"\n{len(failed_files)} gambar GAGAL diproses:")
        for filename, reason in failed_files:
            print(f"  - {filename}  ({reason})")
    else:
        print("\nSemua gambar berhasil diproses, tidak ada yang gagal.")

    if drive_disconnected:
        print("\n>> Mount Drive kena disconnect. Jalankan ulang CELL 1,")
        print(">> lalu jalankan CELL ini lagi (file yang sudah OK tidak perlu diulang).")

#APTOS

In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Box Crop/1b/APTOS"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Box Crop/4b/APTOS"

In [ ]:
process_folder(INPUT_FOLDER, OUTPUT_FOLDER)

In [ ]:
results, failed = preprocess_folder_branch_b(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    reference_image=None,
)

#IDRID


In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Box Crop/1b/IDRID"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Box Crop/4b/IDRID"

In [ ]:
process_folder(INPUT_FOLDER, OUTPUT_FOLDER)

#MESSIDOR

In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Box Crop/1b/MESSIDOR"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Box Crop/4b/MESSIDOR"

In [ ]:
process_folder(INPUT_FOLDER, OUTPUT_FOLDER)